# Shapix Tour

**Elegant runtime shape and dtype checking for NumPy, JAX, and PyTorch arrays.**

Shapix turns array shape annotations into Python objects that [beartype](https://github.com/beartype/beartype) validates at runtime. Dimensions like `N` and `C` are enforced for consistency across all parameters — automatically.

This notebook walks through every major feature.

## 1. Basic usage — shape and dtype checking

Just decorate with `@beartype` and annotate with shapix types. That's it.

In [ ]:
import numpy as np
from beartype import beartype
from shapix import N, C, H, W
from shapix.numpy import F32

@beartype
def normalize(x: F32[N, C]) -> F32[N, C]:
    """Normalize each row to sum to 1."""
    return x / x.sum(axis=1, keepdims=True)

# This works — correct dtype and shape
result = normalize(np.ones((4, 3), dtype=np.float32))
print("Input shape:  (4, 3)")
print(f"Output shape: {result.shape}")
print(f"Row sums:     {result.sum(axis=1)}")  # each row sums to 1

In [ ]:
# Wrong dtype -> caught immediately
try:
    normalize(np.ones((4, 3), dtype=np.float64))
except Exception as e:
    print(f"Wrong dtype: {type(e).__name__}")

# Wrong rank -> caught immediately
try:
    normalize(np.ones((4,), dtype=np.float32))
except Exception as e:
    print(f"Wrong rank:  {type(e).__name__}")

## 2. Cross-argument dimension consistency

Named dimensions are tracked within a function call. If `N` is bound to 4 by the first parameter, every subsequent parameter must agree.

In [ ]:
@beartype
def add(x: F32[N, C], y: F32[N, C]) -> F32[N, C]:
    return x + y

# Same shapes -> OK
result = add(
    np.ones((4, 3), dtype=np.float32),
    np.ones((4, 3), dtype=np.float32),
)
print(f"add((4,3), (4,3)) -> {result.shape}")

# Mismatched N -> caught!
try:
    add(
        np.ones((4, 3), dtype=np.float32),
        np.ones((5, 3), dtype=np.float32),  # N=5 but first arg bound N=4
    )
except Exception as e:
    print(f"add((4,3), (5,3)) -> {type(e).__name__}: N mismatch")

Dimensions can be **shared across different positions** in the signature. Here `C` appears as the second dim of `a` and the first dim of `b`, enforcing that the inner dimensions match for matrix multiplication:

In [ ]:
@beartype
def matmul(a: F32[N, C], b: F32[C, H]) -> F32[N, H]:
    return a @ b

result = matmul(
    np.ones((4, 3), dtype=np.float32),
    np.ones((3, 5), dtype=np.float32),
)
print(f"matmul((4,3), (3,5)) -> {result.shape}")  # (4, 5)

# Inner dim mismatch -> caught
try:
    matmul(
        np.ones((4, 3), dtype=np.float32),
        np.ones((7, 5), dtype=np.float32),  # C=7, but first arg bound C=3
    )
except Exception as e:
    print(f"matmul((4,3), (7,5)) -> C mismatch")

## 3. Sequential calls are independent

Each function invocation gets a fresh set of dimension bindings. Calling the same function twice with different shapes is perfectly fine.

In [ ]:
@beartype
def identity(x: F32[N]) -> F32[N]:
    return x

for size in [3, 100, 7, 42, 1]:
    result = identity(np.ones((size,), dtype=np.float32))
    print(f"identity(shape=({size},)) -> {result.shape}")

## 4. Return type checking

The return value is also validated against the spec, including cross-argument consistency with the input dimensions.

In [ ]:
@beartype
def buggy_function(x: F32[N, C]) -> F32[N, C]:
    """This function has a bug -- it returns the wrong shape!"""
    return np.zeros((1, 1), dtype=np.float32)  # Oops!

try:
    buggy_function(np.ones((4, 3), dtype=np.float32))
except Exception as e:
    print(f"Return type violation caught: {type(e).__name__}")
    print(f"  Expected (4, 3) but function returned (1, 1)")

## 5. Fixed dimensions

Use plain integers for dimensions that must match an exact size:

In [ ]:
@beartype
def process_rgb(image: F32[N, 3, H, W]) -> F32[N, 3, H, W]:
    """Only accepts images with exactly 3 channels."""
    return image / 255.0

# 3 channels -> OK
result = process_rgb(np.ones((2, 3, 28, 28), dtype=np.float32))
print(f"process_rgb((2, 3, 28, 28)) -> {result.shape}")

# 4 channels -> rejected
try:
    process_rgb(np.ones((2, 4, 28, 28), dtype=np.float32))
except Exception as e:
    print(f"process_rgb((2, 4, 28, 28)) -> channel dim must be 3")

## 6. Symbolic dimensions (arithmetic)

Dimensions support Python arithmetic. Expressions are evaluated against bound dimension values at runtime.

In [ ]:
@beartype
def pad_both_sides(x: F32[N]) -> F32[N + 2]:
    """Pad a 1D array with one zero on each side."""
    return np.pad(x, 1)

result = pad_both_sides(np.ones((5,), dtype=np.float32))
print(f"pad_both_sides(shape=(5,)) -> {result.shape}")  # (7,) = 5+2

@beartype
def flatten(x: F32[N, C]) -> F32[N * C]:
    """Flatten a 2D array into 1D."""
    return x.reshape(-1)

result = flatten(np.ones((3, 4), dtype=np.float32))
print(f"flatten(shape=(3, 4)) -> {result.shape}")  # (12,) = 3*4

# Wrong output shape -> caught
@beartype
def bad_pad(x: F32[N]) -> F32[N + 2]:
    return np.pad(x, 2)  # Pads 2 on each side -> N+4, not N+2!

try:
    bad_pad(np.ones((5,), dtype=np.float32))
except Exception as e:
    print(f"bad_pad: expected shape (7,) but got (9,)")

## 7. Variadic dimensions

Apply `~` (tilde) to any dimension to make it **variadic** — matching **zero or more** contiguous dimensions. This is perfect for batch dimensions that may or may not be present.

In [ ]:
from shapix import B, _

@beartype
def softmax(x: F32[~B, C]) -> F32[~B, C]:
    """Softmax along the last dim, accepting any number of leading batch dims."""
    exp_x = np.exp(x - x.max(axis=-1, keepdims=True))
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

# No batch dim
print(f"softmax((3,))       -> {softmax(np.ones((3,), dtype=np.float32)).shape}")
# One batch dim
print(f"softmax((4, 3))     -> {softmax(np.ones((4, 3), dtype=np.float32)).shape}")
# Two batch dims
print(f"softmax((2, 4, 3))  -> {softmax(np.ones((2, 4, 3), dtype=np.float32)).shape}")

Named variadic dims enforce cross-argument consistency on the batch shape. Use `~_` for anonymous variadics when you don't need consistency.

In [ ]:
@beartype
def batch_add(x: F32[~B, C], y: F32[~B, C]) -> F32[~B, C]:
    """Both inputs must have the same batch dimensions."""
    return x + y

# Same batch -> OK
result = batch_add(
    np.ones((2, 4, 3), dtype=np.float32),
    np.ones((2, 4, 3), dtype=np.float32),
)
print(f"batch_add((2,4,3), (2,4,3)) -> {result.shape}")

# Different batch -> caught
try:
    batch_add(
        np.ones((2, 4, 3), dtype=np.float32),
        np.ones((5, 4, 3), dtype=np.float32),  # batch (5,4) != (2,4)
    )
except Exception as e:
    print(f"batch_add((2,4,3), (5,4,3)) -> batch mismatch")

## 8. Broadcastable dimensions

Apply `+` (unary plus) to any dimension to make it **broadcastable** — size 1 always matches, following NumPy broadcasting rules:

In [ ]:
@beartype
def broadcast_add(x: F32[N, C], y: F32[+N, C]) -> F32[N, C]:
    """y's first dim can be 1 (broadcast) or match N."""
    return x + y

# Normal match
result = broadcast_add(np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32))
print(f"broadcast_add((4,3), (4,3)) -> {result.shape}")

# Broadcast: +N=1
result = broadcast_add(np.ones((4, 3), dtype=np.float32), np.ones((1, 3), dtype=np.float32))
print(f"broadcast_add((4,3), (1,3)) -> {result.shape}")

## 9. Anonymous dimensions

`_` matches any single dimension without binding — useful when you don't care about a particular dimension's consistency.

In [ ]:
@beartype
def sum_last_dim(x: F32[_, _, C]) -> F32[_, _]:
    """We only care that the last dim is consistent. First two are anonymous."""
    return x.sum(axis=-1)

result = sum_last_dim(np.ones((4, 3, 5), dtype=np.float32))
print(f"sum_last_dim((4, 3, 5)) -> {result.shape}")

result = sum_last_dim(np.ones((99, 1, 5), dtype=np.float32))
print(f"sum_last_dim((99, 1, 5)) -> {result.shape}  (anonymous dims match anything)")

## 10. Custom dimensions

Create your own dimension symbols with `Dimension` for domain-specific annotations:

In [ ]:
from shapix import Dimension
from shapix.numpy import I64

# Domain-specific dimensions
Vocab = Dimension("Vocab")
Embed = Dimension("Embed")
Seq = Dimension("Seq")

@beartype
def embed_tokens(
    tokens: I64[N, Seq],
    table: F32[Vocab, Embed],
) -> F32[N, Seq, Embed]:
    """Look up token embeddings from an embedding table."""
    return table[tokens]

tokens = np.array([[0, 1, 2], [3, 0, 1]], dtype=np.int64)  # (2, 3)
table = np.random.randn(5, 8).astype(np.float32)            # (5, 8) -> Vocab=5, Embed=8
result = embed_tokens(tokens, table)
print(f"embed_tokens: tokens{tokens.shape} + table{table.shape} -> {result.shape}")
print(f"  N=2, Seq=3, Vocab=5, Embed=8")

## 11. Nested function calls

Each beartype-decorated function call gets its own dimension memo, so nested calls are fully independent:

In [ ]:
@beartype
def inner(x: F32[N]) -> F32[N]:
    """N means something different here than in the outer function."""
    return x * 2

@beartype
def outer(x: F32[N, C]) -> F32[N]:
    """Call inner with just the first column -- different rank!"""
    first_col = x[:, 0]  # shape (N,)
    return inner(first_col)

result = outer(np.ones((4, 3), dtype=np.float32))
print(f"outer((4, 3)) calls inner((4,)) -> {result.shape}")
print(f"  N=4 in outer's context, N=4 in inner's context (independent memos)")

## 12. Multiple dtype arrays

Shapix supports all common dtypes and category groups. You can mix different dtypes in one function signature:

In [ ]:
from shapix.numpy import F64, I32, Float, Shaped

@beartype
def weighted_sum(
    values: F32[N, C],
    indices: I32[N],
) -> F64[C]:
    """Select rows by index and sum, returning float64."""
    selected = values[indices]
    return selected.sum(axis=0).astype(np.float64)

values = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
indices = np.array([0, 2, 0], dtype=np.int32)
result = weighted_sum(values, indices)
print(f"weighted_sum: values{values.shape} + indices{indices.shape} -> {result.shape}")
print(f"  Result dtype: {result.dtype}")

# Float accepts any float dtype
@beartype
def any_float(x: Float[N]) -> Float[N]:
    return x

print(f"\nFloat accepts float32: {any_float(np.ones(3, dtype=np.float32)).dtype}")
print(f"Float accepts float64: {any_float(np.ones(3, dtype=np.float64)).dtype}")
print(f"Float accepts float16: {any_float(np.ones(3, dtype=np.float16)).dtype}")

## 13. Custom array types

Use `make_array_type` and `DtypeSpec` to create types for custom array classes or custom dtype combinations:

In [ ]:
from shapix import make_array_type, DtypeSpec

# Custom dtype: only float32 or float16 (e.g. for mixed-precision training)
MIXED_PRECISION = DtypeSpec("MixedPrecision", frozenset({"float16", "float32"}))
MixedArray = make_array_type(np.ndarray, MIXED_PRECISION)

@beartype
def mixed_op(x: MixedArray[N, C]) -> MixedArray[N, C]:
    return x

# float32 -> OK
result = mixed_op(np.ones((4, 3), dtype=np.float32))
print(f"MixedArray accepts float32: {result.dtype}")

# float16 -> OK
result = mixed_op(np.ones((4, 3), dtype=np.float16))
print(f"MixedArray accepts float16: {result.dtype}")

# float64 -> rejected
try:
    mixed_op(np.ones((4, 3), dtype=np.float64))
except Exception as e:
    print(f"MixedArray rejects float64")

## 14. Explicit memo management

For advanced use cases or guaranteed correctness, use `@shapix.check` or `check_context`:

In [ ]:
import shapix
from beartype.door import is_bearable

# @shapix.check decorator -- explicit memo push/pop around the call
@shapix.check
@beartype
def checked_add(x: F32[N, C], y: F32[N, C]) -> F32[N, C]:
    return x + y

result = checked_add(np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32))
print(f"@shapix.check + @beartype: {result.shape}")

# check_context -- for manual is_bearable() checks with shared dimensions
with shapix.check_context():
    x = np.ones((4, 3), dtype=np.float32)
    y = np.ones((4, 5), dtype=np.float32)

    result1 = is_bearable(x, F32[N, C])
    result2 = is_bearable(y, F32[N, H])  # N=4 from x, H=5 from y

    print(f"\ncheck_context:")
    print(f"  x(4,3) matches F32[N, C]: {result1}")
    print(f"  y(4,5) matches F32[N, H]: {result2}  (N=4 consistent, H=5 binds)")

# Outside the context, dimensions are fresh
with shapix.check_context():
    z = np.ones((10, 7), dtype=np.float32)
    result3 = is_bearable(z, F32[N, C])
    print(f"  z(10,7) in new context matches F32[N, C]: {result3}  (N=10, fresh memo)")

## 15. Putting it all together — a mini neural network forward pass

Here's a realistic example combining multiple features:

In [ ]:
from shapix import B, Dimension

# Domain-specific dimensions
Features = Dimension("Features")
Hidden = Dimension("Hidden")
Classes = Dimension("Classes")

@beartype
def linear(x: F32[B, Features], weight: F32[Features, Hidden]) -> F32[B, Hidden]:
    """A single linear layer: y = x @ W."""
    return x @ weight

@beartype
def relu(x: F32[B, Hidden]) -> F32[B, Hidden]:
    """ReLU activation."""
    return np.maximum(x, 0)

@beartype
def classifier(
    x: F32[B, Features],
    w1: F32[Features, Hidden],
    w2: F32[Hidden, Classes],
) -> F32[B, Classes]:
    """Two-layer classifier: linear → relu → linear."""
    h = linear(x, w1)
    h = relu(h)
    # For the second linear layer, we need different dim names
    logits = h @ w2
    return logits

# Set up dimensions
batch, features, hidden, classes = 8, 10, 32, 5
x = np.random.randn(batch, features).astype(np.float32)
w1 = np.random.randn(features, hidden).astype(np.float32)
w2 = np.random.randn(hidden, classes).astype(np.float32)

logits = classifier(x, w1, w2)
print(f"✓ classifier:")
print(f"  Input:   {x.shape}  (B={batch}, Features={features})")
print(f"  W1:      {w1.shape} (Features={features}, Hidden={hidden})")
print(f"  W2:      {w2.shape} (Hidden={hidden}, Classes={classes})")
print(f"  Output:  {logits.shape}  (B={batch}, Classes={classes})")
print(f"\n  All dimensions verified: Features matches across x and w1,")
print(f"  Hidden matches across w1 output and w2 input — automatically!")

## Dimension cheat sheet

| Syntax | Meaning | Example |
|--------|---------|---------|
| `N` | Named dim — bind & enforce | `F32[N, C]` |
| `3` | Fixed dim — exact match | `F32[3, N]` |
| `N + 1` | Symbolic — arithmetic expression | `F32[N + 1]` |
| `~B` | Variadic — zero or more dims | `F32[~B, C]` |
| `+N` | Broadcastable — size 1 OK | `F32[+N, C]` |
| `_` | Anonymous — any size, no binding | `F32[_, C]` |
| `~_` | Anonymous variadic — any dims | `F32[~_, C]` |

**Array types by backend:**
- NumPy: `shapix.numpy.F32`, `I64`, `Shaped`, ...
- JAX: `shapix.jax.F32`, `BF16`, ...
- PyTorch: `shapix.torch.F32`, `BF16`, ...

**Like types** (scalar | array | nested sequence):
- `shapix.numpy.F32Like`, `I64Like`, `ArrayLike`, ...